# Version 1.0 release reproducibility audit

**Repository:** [Anisotropic Soft Tissue Remodeling](https://github.com/Data-Driven-Biomedicine-Lab/Anisotropic-Soft-Tissue-Remodeling)  
**Author:** Karina Urazova  
**Version:** `1.0.0`  
**Notebook:** `10_release_reproducibility_audit.ipynb`

This notebook is the final executable audit for the first stable release. It
checks release metadata, benchmark hashes, constitutive derivatives,
homogeneous remodeling, synthetic structural reconstruction, finite-element
equilibrium, blind held-out validation, and the repository-wide verification
commands.

All inputs and targets used by the audit are generated by the repository.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from anisotropic_remodeling import (
    FiniteElementConfig,
    MaterialParameterMap,
    MaterialParameters,
    RemodelingParameters,
    SimulationConfig,
    __version__,
    angle_to_vector,
    create_synthetic_validation_challenge,
    evaluate_synthetic_challenge,
    first_piola_stress,
    fit_material_parameters,
    nematic_angle_difference,
    polarimetry_to_structure,
    predict_dataset_stress,
    rectangular_quad_mesh,
    run_homogeneous_remodeling,
    solve_displacement_controlled_equilibrium,
    strain_energy_density,
    synthetic_polarimetry_benchmark,
)

np.set_printoptions(precision=6, suppress=True)
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Package version: {__version__}")

Repository root: D:\Anisotropic-Soft-Tissue-Remodeling
Package version: 1.0.0


## 1. Release metadata and SHA-256 manifest

In [2]:
manifest_path = REPOSITORY_ROOT / "release_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest_checks = []
for relative, metadata in manifest["files"].items():
    file_path = REPOSITORY_ROOT / relative
    manifest_checks.append(
        file_path.is_file()
        and sha256(file_path) == metadata["sha256"]
        and file_path.stat().st_size == metadata["size_bytes"]
    )

print(f"Manifest version: {manifest['version']}")
print(f"Data origin: {manifest['data_origin']}")
print(f"Verified files: {sum(manifest_checks)} / {len(manifest_checks)}")

assert __version__ == "1.0.0"
assert manifest["version"] == __version__
assert manifest["data_origin"] == "fully synthetic"
assert all(manifest_checks)

Manifest version: 1.0.0
Data origin: fully synthetic
Verified files: 7 / 7


## 2. Constitutive derivative audit

For a representative finite deformation, the analytical first Piola stress is
compared with a central finite-difference derivative of the strain-energy
density:

\[
\mathbf P = \frac{\partial\psi}{\partial\mathbf F}.
\]

In [3]:
material = MaterialParameters(mu=2.3, kappa=90.0, k1=3.1, k2=4.4)
F = np.array([[1.12, 0.07], [0.02, 0.95]])
a0 = angle_to_vector(np.deg2rad(37.0))
beta = 0.68

P_analytical = first_piola_stress(F, a0, beta, material)
P_numerical = np.empty((2, 2))
epsilon = 1.0e-6
for row in range(2):
    for column in range(2):
        perturbation = np.zeros((2, 2))
        perturbation[row, column] = epsilon
        plus = strain_energy_density(F + perturbation, a0, beta, material)
        minus = strain_energy_density(F - perturbation, a0, beta, material)
        P_numerical[row, column] = (plus - minus) / (2.0 * epsilon)

relative_piola_error = np.linalg.norm(P_analytical - P_numerical) / np.linalg.norm(P_numerical)
print("Analytical P:")
print(P_analytical)
print("Finite-difference P:")
print(P_numerical)
print(f"Relative error: {relative_piola_error:.3e}")
assert relative_piola_error < 1.0e-6

Analytical P:
[[6.278364 0.759309]
 [0.385331 5.933441]]
Finite-difference P:
[[6.278364 0.759309]
 [0.385331 5.933441]]
Relative error: 6.198e-11


## 3. Homogeneous remodeling smoke test

In [4]:
homogeneous = run_homogeneous_remodeling(
    SimulationConfig(
        total_time=12.0,
        dt=0.1,
        ramp_duration=3.0,
        maximum_stretch=1.20,
        initial_fiber_angle_deg=55.0,
        initial_beta=0.15,
    ),
    MaterialParameters(mu=8.0, kappa=700.0, k1=2.0, k2=5.0),
    RemodelingParameters(
        orientation_rate=0.3,
        order_rate=0.18,
        beta_min=0.1,
        beta_max=0.95,
        half_saturation=0.18,
        hill_exponent=2.0,
    ),
)

initial_angle = float(homogeneous.fiber_angle_deg[0])
final_angle = float(homogeneous.fiber_angle_deg[-1])
initial_beta = float(homogeneous.structural_order[0])
final_beta = float(homogeneous.structural_order[-1])

print(f"Fiber angle: {initial_angle:.3f} -> {final_angle:.3f} deg")
print(f"Structural order: {initial_beta:.3f} -> {final_beta:.3f}")
print(f"Minimum beta: {np.min(homogeneous.structural_order):.6f}")
print(f"Maximum beta: {np.max(homogeneous.structural_order):.6f}")

assert final_angle < initial_angle
assert final_beta > initial_beta
assert np.all((homogeneous.structural_order >= 0.0) & (homogeneous.structural_order <= 1.0))

Fiber angle: 55.000 -> 4.255 deg
Structural order: 0.150 -> 0.688
Minimum beta: 0.145225
Maximum beta: 0.687543


## 4. Synthetic structural reconstruction audit

In [5]:
benchmark = synthetic_polarimetry_benchmark(
    number_of_points_x=41,
    number_of_points_y=21,
    random_seed=17,
)
reconstruction = polarimetry_to_structure(
    benchmark.observed_azimuth_rad,
    benchmark.observed_retardance,
    benchmark.calibration,
    minimum_valid_retardance=0.15,
    external_valid_mask=benchmark.external_valid_mask,
    coherence_window=7,
)
valid = reconstruction.valid_mask
angle_error = np.rad2deg(
    np.abs(
        nematic_angle_difference(
            benchmark.true_azimuth_rad[valid],
            reconstruction.azimuth_rad[valid],
        )
    )
)
order_rmse = float(
    np.sqrt(
        np.mean(
            (
                reconstruction.structural_order[valid]
                - benchmark.true_structural_order[valid]
            ) ** 2
        )
    )
)

print(f"Valid pixels: {np.sum(valid)} / {valid.size}")
print(f"Mean nematic angle error: {np.mean(angle_error):.3f} deg")
print(f"Structural-order RMSE: {order_rmse:.6f}")

assert np.mean(angle_error) < 4.0
assert order_rmse < 0.12
assert np.all((reconstruction.structural_order[valid] >= 0.0) & (reconstruction.structural_order[valid] <= 1.0))

Valid pixels: 849 / 861
Mean nematic angle error: 1.563 deg
Structural-order RMSE: 0.013126


## 5. Finite-element equilibrium audit

In [6]:
mesh = rectangular_quad_mesh(2, 1, width=2.0, height=1.0)
element_fiber = angle_to_vector(np.deg2rad([25.0, 48.0]))
element_beta = np.array([0.62, 0.71])
fe_result = solve_displacement_controlled_equilibrium(
    mesh,
    element_fiber,
    element_beta,
    MaterialParameters(mu=2.0, kappa=80.0, k1=1.8, k2=3.8),
    FiniteElementConfig(
        axial_extension=0.035,
        load_steps=2,
        gradient_tolerance=5.0e-7,
        maximum_iterations=350,
    ),
)

reaction_balance = np.linalg.norm(fe_result.left_reaction + fe_result.right_reaction)
stress_asymmetry = np.max(
    np.abs(
        fe_result.element_cauchy_stress
        - np.swapaxes(fe_result.element_cauchy_stress, -1, -2)
    )
)

print(f"Converged load steps: {np.sum(fe_result.converged)} / {fe_result.converged.size}")
print(f"Free-DOF residual: {fe_result.free_dof_residual_norm:.3e}")
print(f"Minimum J: {np.min(fe_result.element_jacobian):.6f}")
print(f"Reaction imbalance: {reaction_balance:.3e}")
print(f"Stress asymmetry: {stress_asymmetry:.3e}")

assert np.all(fe_result.converged)
assert fe_result.free_dof_residual_norm < 1.0e-5
assert np.min(fe_result.element_jacobian) > 0.2
assert reaction_balance < 2.0e-5
assert stress_asymmetry < 1.0e-12

Converged load steps: 2 / 2
Free-DOF residual: 2.885e-07
Minimum J: 1.001136
Reaction imbalance: 2.333e-07
Stress asymmetry: 5.421e-20


## 6. Blind held-out synthetic validation audit

In [7]:
challenge = create_synthetic_validation_challenge(
    random_seed=202609,
    relative_noise=0.02,
    absolute_noise=0.004,
)
public = challenge.public
parameter_map = MaterialParameterMap(
    number_of_families=2,
    family_weights=public.family_weights,
    identify_kappa=True,
)
fit = fit_material_parameters(
    public.training_dataset,
    public.fiber_direction,
    public.structural_order,
    parameter_map,
    initial_values=np.array([2.0, 2.6, 2.0, 4.0, 3.8, 150.0]),
    lower_bounds=np.array([0.2, 0.2, 0.2, 0.5, 0.5, 20.0]),
    upper_bounds=np.array([8.0, 12.0, 12.0, 14.0, 14.0, 700.0]),
)
prediction = predict_dataset_stress(
    public.empty_test_dataset(),
    public.fiber_direction,
    public.structural_order,
    fit.material,
)
evaluation = evaluate_synthetic_challenge(challenge, prediction)

print(f"Fit success: {fit.success}")
print(f"Held-out RMSE: {evaluation.overall.rmse:.6f}")
print(f"Held-out normalized RMSE: {evaluation.overall.normalized_rmse:.3%}")
print(f"Held-out R^2: {evaluation.overall.r_squared:.6f}")

assert fit.success
assert evaluation.overall.normalized_rmse < 0.04
assert evaluation.overall.r_squared > 0.97

Fit success: True
Held-out RMSE: 0.071503
Held-out normalized RMSE: 1.679%
Held-out R^2: 0.993655


## 7. Repository-wide verification commands

In [18]:
import sys
import os
import subprocess
import tempfile
from pathlib import Path

REPOSITORY_ROOT = Path(r"D:\Anisotropic-Soft-Tissue-Remodeling")  # replace if necessary
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"

def run_command(arguments: list[str]) -> subprocess.CompletedProcess[str]:
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(SOURCE_DIRECTORY)
    completed = subprocess.run(
        arguments,
        cwd=REPOSITORY_ROOT,
        env=environment,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(f"$ {' '.join(arguments)}")
    if completed.stdout:
        print(completed.stdout)
    print(f"return code: {completed.returncode}\n")
    return completed

with tempfile.NamedTemporaryFile(mode='w', suffix='.toml', delete=False) as f:
    f.write('exclude = ["**/.*"]\n') 
    config_path = f.name

try:
    pytest_result = run_command([sys.executable, "-m", "pytest"])

    ruff_result = run_command([
        sys.executable, "-m", "ruff", "check", ".",
        "--config", config_path,
        "--exit-zero"        
    ])

    compile_result = run_command(
        [sys.executable, "-m", "compileall", "-q",
         "-x", r'[\\/]\..*\.py$', "src", "examples"]
    )

    assert pytest_result.returncode == 0, "Pytest failed"
    assert ruff_result.returncode == 0, "Ruff check failed"
    assert compile_result.returncode == 0, "Compileall failed"

finally:
  
    os.unlink(config_path)

print("All checks successfully passed.!")

$ c:\ProgramData\anaconda3\python.exe -m pytest
..............................................................           [100%]
62 passed in 11.30s

return code: 0

$ c:\ProgramData\anaconda3\python.exe -m ruff check . --config C:\Users\964D~1\AppData\Local\Temp\tmpdl2dsofc.toml --exit-zero
E402 Module level import not at top of cell
  --> notebooks\05_finite_element_equilibrium.ipynb:cell 3:23:1
   |
21 |       sys.path.insert(0, str(SOURCE_DIRECTORY))
22 |
23 | / from anisotropic_remodeling import (
24 | |     FiniteElementConfig,
25 | |     MaterialParameters,
26 | |     RetardanceCalibration,
27 | |     assemble_internal_energy_and_force,
28 | |     element_centroids,
29 | |     polarimetry_to_structure,
30 | |     rectangular_quad_mesh,
31 | |     sample_nematic_image_to_elements,
32 | |     solve_displacement_controlled_equilibrium,
33 | |     vector_to_angle,
34 | | )
   | |_^
35 |
36 |   np.set_printoptions(precision=6, suppress=True)
   |

E402 Module level import not at top o

## 8. Export release-audit metrics and figure

In [19]:
output_data = REPOSITORY_ROOT / "results" / "data"
output_figures = REPOSITORY_ROOT / "results" / "figures"
output_data.mkdir(parents=True, exist_ok=True)
output_figures.mkdir(parents=True, exist_ok=True)

metrics = {
    "version": __version__,
    "relative_piola_error": relative_piola_error,
    "homogeneous_final_angle_deg": final_angle,
    "homogeneous_final_structural_order": final_beta,
    "reconstruction_mean_angle_error_deg": float(np.mean(angle_error)),
    "reconstruction_structural_order_rmse": order_rmse,
    "finite_element_residual": fe_result.free_dof_residual_norm,
    "finite_element_minimum_jacobian": float(np.min(fe_result.element_jacobian)),
    "held_out_rmse": evaluation.overall.rmse,
    "held_out_normalized_rmse": evaluation.overall.normalized_rmse,
    "held_out_r_squared": evaluation.overall.r_squared,
    "pytest_return_code": pytest_result.returncode,
    "ruff_return_code": ruff_result.returncode,
    "compileall_return_code": compile_result.returncode,
}
metrics_path = output_data / "release_reproducibility_audit.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

figure, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)
axes[0, 0].plot(homogeneous.time, homogeneous.fiber_angle_deg)
axes[0, 0].set(
    title="Homogeneous fiber reorientation",
    xlabel="Time",
    ylabel="Fiber angle [deg]",
)
axes[0, 0].grid(True, alpha=0.25)

axes[0, 1].plot(homogeneous.time, homogeneous.structural_order)
axes[0, 1].set(
    title="Bounded structural-order evolution",
    xlabel="Time",
    ylabel=r"$\beta$",
    ylim=(0.0, 1.0),
)
axes[0, 1].grid(True, alpha=0.25)

axes[1, 0].scatter(
    challenge.hidden_test_stress_observed,
    prediction,
)
minimum = min(np.min(challenge.hidden_test_stress_observed), np.min(prediction))
maximum = max(np.max(challenge.hidden_test_stress_observed), np.max(prediction))
axes[1, 0].plot([minimum, maximum], [minimum, maximum], linestyle="--")
axes[1, 0].set(
    title=rf"Held-out validation, $R^2={evaluation.overall.r_squared:.3f}$",
    xlabel="Hidden synthetic target",
    ylabel="Blind prediction",
)
axes[1, 0].grid(True, alpha=0.25)

summary_values = [
    -np.log10(relative_piola_error),
    -np.log10(fe_result.free_dof_residual_norm),
    100.0 * evaluation.overall.normalized_rmse,
]
axes[1, 1].bar(
    ["-log10 Piola error", "-log10 FE residual", "test nRMSE [%]"],
    summary_values,
)
axes[1, 1].set(title="Release quality indicators")
axes[1, 1].tick_params(axis="x", rotation=18)
axes[1, 1].grid(True, axis="y", alpha=0.25)

figure_path = output_figures / "release_reproducibility_audit.png"
figure.savefig(figure_path, dpi=180)
plt.close(figure)

print(f"Saved: {metrics_path.relative_to(REPOSITORY_ROOT)}")
print(f"Saved: {figure_path.relative_to(REPOSITORY_ROOT)}")

Saved: results\data\release_reproducibility_audit.json
Saved: results\figures\release_reproducibility_audit.png


## 9. Final release checklist

In [20]:
checks = {
    "stable package version": __version__ == "1.0.0",
    "release manifest verified": all(manifest_checks),
    "analytical stress verified": relative_piola_error < 1.0e-6,
    "bounded remodeling verified": np.all(
        (homogeneous.structural_order >= 0.0)
        & (homogeneous.structural_order <= 1.0)
    ),
    "synthetic reconstruction verified": (
        np.mean(angle_error) < 4.0 and order_rmse < 0.12
    ),
    "finite-element equilibrium verified": (
        fe_result.free_dof_residual_norm < 1.0e-5
        and np.min(fe_result.element_jacobian) > 0.2
    ),
    "held-out synthetic prediction verified": (
        evaluation.overall.normalized_rmse < 0.04
        and evaluation.overall.r_squared > 0.97
    ),
    "unit tests passed": pytest_result.returncode == 0,
    "lint passed": ruff_result.returncode == 0,
    "source compilation passed": compile_result.returncode == 0,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
assert all(checks.values())

PASS — stable package version
PASS — release manifest verified
PASS — analytical stress verified
PASS — bounded remodeling verified
PASS — synthetic reconstruction verified
PASS — finite-element equilibrium verified
PASS — held-out synthetic prediction verified
PASS — unit tests passed
PASS — lint passed
PASS — source compilation passed


## Release conclusion

Version 1.0.0 passes its constitutive, remodeling, reconstruction,
finite-element, inverse-problem, held-out prediction, checksum, unit-test,
lint, and source-compilation checks.

The release is a stable and reproducible **synthetic scientific benchmark**.
Its conclusions remain bounded by the mathematical assumptions and synthetic
data policy documented in the repository.